# Protein ProGen Pretraining From Scratch

This notebook trains a protein-only MDC decoder from random initialization with the repo pretrain helpers in libs/core.

It supports:
- single GPU or CPU by default;
- automatic DataParallel when multiple local GPUs are visible in the notebook session;
- DDP when the kernel is launched under torchrun and WORLD_SIZE, RANK, and LOCAL_RANK are set.

The corpus still expects compiled `<|protein|>...<|endoftext|>` lines from the stage-2 data pipeline.

In [2]:
import json
import math
import os
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    MDCDecoderModel,
    build_mdc_config_from_progen_config,
    build_or_load_protein_tokenizer,
    build_or_load_protein_tokenizer_from_text_paths,
    build_progen_config,
    compute_mdc_causal_lm_loss,
    count_trainable_parameters,
    create_protein_lm_dataloader,
    create_streaming_protein_lm_dataloader,
    discover_protein_train_text_paths,
    evaluate_mdc_causal_lm_batch_loss,
    generate_protein_text,
    load_protein_corpus_text_parts,
    prepare_mdc_training_runtime,
    save_protein_pretrain_checkpoint,
    set_mdc_data_loader_epoch,
    split_protein_corpus_text,
)
from libs.core.pretrain.training_config import (
    build_protein_training_data_config,
    create_protein_training_optimizer,
    describe_protein_training_optimizers,
    load_protein_training_config,
)

PROJECT_ROOT

WindowsPath('C:/Users/ekko.huynh/Microbial DNA Compiler')

In [ ]:
TRAINING_CONFIG = load_protein_training_config(PROJECT_ROOT)
PATHS_CONFIG = TRAINING_CONFIG["paths"]
DATA_CONFIG = TRAINING_CONFIG["data"]
MODEL_CONFIG = TRAINING_CONFIG["model"]
TRAINING_SETTINGS = TRAINING_CONFIG["training"]
OPTIMIZER_CONFIG = TRAINING_CONFIG["optimizer"]
RUNTIME_CONFIG = TRAINING_CONFIG["runtime"]
RESUME_CONFIG = TRAINING_CONFIG["resume"]
MINIO_CONFIG = TRAINING_CONFIG["minio"]
MINIO_DATA_CONFIG = build_protein_training_data_config(PROJECT_ROOT, TRAINING_CONFIG)

TRAIN_TEXT_PATH = PATHS_CONFIG["train_text_path"]
TOKENIZER_MAP_PATH = PATHS_CONFIG["tokenizer_map_path"]
CHECKPOINT_DIR = PATHS_CONFIG["checkpoint_dir"]
TRAIN_PART_CACHE_DIR = PATHS_CONFIG["train_part_cache_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PART_GLOB = DATA_CONFIG["train_part_glob"]
PREFER_LOCAL_TRAIN_PARTS = DATA_CONFIG["prefer_local_train_parts"]
STREAM_LOCAL_TRAIN_PARTS = DATA_CONFIG["stream_local_train_parts"]
KEEP_DOWNLOADED_TRAIN_PARTS = DATA_CONFIG["keep_downloaded_train_parts"]
TRAIN_RATIO = DATA_CONFIG["train_ratio"]
BATCH_SIZE = DATA_CONFIG["batch_size"]
NUM_WORKERS = DATA_CONFIG["num_workers"]
PIN_MEMORY = DATA_CONFIG["pin_memory"]

MINIO_TRAIN_PARTS_PREFIX_URI = MINIO_CONFIG["train_parts_prefix_uri"]
MINIO_TRAIN_PART_URIS = MINIO_CONFIG["train_part_uris"]

PROGEN_MODEL_SIZE = MODEL_CONFIG["progen_model_size"]
PROGEN_CONFIG_OVERRIDES = MODEL_CONFIG["progen_config_overrides"]
CONTEXT_LENGTH = MODEL_CONFIG["context_length"]
STRIDE = MODEL_CONFIG["stride"]
TOKENIZER_VOCAB_SIZE = MODEL_CONFIG["tokenizer_vocab_size"]
REBUILD_TOKENIZER = MODEL_CONFIG["rebuild_tokenizer"]

NUM_EPOCHS = TRAINING_SETTINGS["num_epochs"]
GRAD_CLIP_NORM = TRAINING_SETTINGS["grad_clip_norm"]
EVAL_FREQ = TRAINING_SETTINGS["eval_freq"]
EVAL_BATCHES = TRAINING_SETTINGS["eval_batches"]

OPTIMIZER_TYPE = OPTIMIZER_CONFIG["type"]
LEARNING_RATE = OPTIMIZER_CONFIG["learning_rate"]
MUON_LEARNING_RATE = OPTIMIZER_CONFIG["muon_learning_rate"]
WEIGHT_DECAY = OPTIMIZER_CONFIG["weight_decay"]

REQUESTED_DEVICE = RUNTIME_CONFIG["device"]
MULTI_GPU_MODE = RUNTIME_CONFIG["multi_gpu_mode"]
DDP_FIND_UNUSED_PARAMETERS = RUNTIME_CONFIG["ddp_find_unused_parameters"]
DATA_PARALLEL_DEVICE_IDS = RUNTIME_CONFIG["data_parallel_device_ids"]

RANK = int(os.environ.get("RANK", "0"))
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", os.environ.get("RANK", "0")))
WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))
IS_DISTRIBUTED = WORLD_SIZE > 1
IS_MAIN_PROCESS = RANK == 0

USE_MINIO_TRAIN_PARTS = bool(MINIO_TRAIN_PARTS_PREFIX_URI or MINIO_TRAIN_PART_URIS)
LOCAL_TRAIN_TEXT_PATHS = ()
if TRAIN_TEXT_PATH.exists() or any(TRAIN_TEXT_PATH.parent.glob(TRAIN_PART_GLOB)):
    LOCAL_TRAIN_TEXT_PATHS = discover_protein_train_text_paths(
        TRAIN_TEXT_PATH,
        part_glob=TRAIN_PART_GLOB,
        prefer_parts=PREFER_LOCAL_TRAIN_PARTS,
    )
elif not USE_MINIO_TRAIN_PARTS:
    raise FileNotFoundError(
        f"Missing protein corpus: {TRAIN_TEXT_PATH} or {TRAIN_PART_GLOB} parts. "
        "Build it first with cmd/build_refseq_profile_text.py."
    )
elif not TOKENIZER_MAP_PATH.exists():
    raise FileNotFoundError(
        "When streaming training parts from MinIO without a local train.txt/train_part_*.txt corpus, "
        f"provide the tokenizer map first: {TOKENIZER_MAP_PATH}"
    )

config_summary = {
    "train_config_path": str(TRAINING_CONFIG["config_path"]),
    "train_text_path": str(TRAIN_TEXT_PATH),
    "local_train_text_paths": [str(path) for path in LOCAL_TRAIN_TEXT_PATHS],
    "tokenizer_map_path": str(TOKENIZER_MAP_PATH),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "train_part_cache_dir": str(TRAIN_PART_CACHE_DIR),
    "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
    "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
    "minio_endpoint_url": MINIO_DATA_CONFIG.minio.normalized_endpoint_url if MINIO_DATA_CONFIG is not None else None,
    "stream_local_train_parts": STREAM_LOCAL_TRAIN_PARTS,
    "progen_model_size": PROGEN_MODEL_SIZE,
    "progen_config_overrides": PROGEN_CONFIG_OVERRIDES,
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "tokenizer_vocab_size": TOKENIZER_VOCAB_SIZE,
    "optimizer_type": OPTIMIZER_TYPE,
    "learning_rate": LEARNING_RATE,
    "muon_learning_rate": MUON_LEARNING_RATE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "multi_gpu_mode": MULTI_GPU_MODE,
    "rank": RANK,
    "local_rank": LOCAL_RANK,
    "world_size": WORLD_SIZE,
}
config_summary

In [ ]:
use_streaming_loader = USE_MINIO_TRAIN_PARTS or (STREAM_LOCAL_TRAIN_PARTS and len(LOCAL_TRAIN_TEXT_PATHS) > 1)
if use_streaming_loader and (REBUILD_TOKENIZER or not TOKENIZER_MAP_PATH.exists()):
    raise ValueError(
        "Streaming large protein corpora requires an existing tokenizer_map.json. "
        "Build it once on a bounded sample or set model.rebuild_tokenizer=false with a valid map."
    )

corpus_text = ""
if not use_streaming_loader and LOCAL_TRAIN_TEXT_PATHS:
    corpus_text = load_protein_corpus_text_parts(LOCAL_TRAIN_TEXT_PATHS)

if LOCAL_TRAIN_TEXT_PATHS:
    tokenizer_artifact = build_or_load_protein_tokenizer_from_text_paths(
        LOCAL_TRAIN_TEXT_PATHS,
        tokenizer_map_path=TOKENIZER_MAP_PATH,
        vocab_size=TOKENIZER_VOCAB_SIZE,
        rebuild=REBUILD_TOKENIZER,
    )
else:
    tokenizer_artifact = build_or_load_protein_tokenizer(
        TRAIN_TEXT_PATH,
        tokenizer_map_path=TOKENIZER_MAP_PATH,
        vocab_size=TOKENIZER_VOCAB_SIZE,
        rebuild=REBUILD_TOKENIZER,
    )

if corpus_text:
    train_text, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO)
else:
    train_text, val_text = "", ""

loader_kwargs = {
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}
distributed_loader_kwargs = {
    "distributed": IS_DISTRIBUTED,
    "rank": RANK,
    "world_size": WORLD_SIZE,
}

if USE_MINIO_TRAIN_PARTS:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer_artifact.tokenizer,
        prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
        part_uris=MINIO_TRAIN_PART_URIS or None,
        config=MINIO_DATA_CONFIG,
        cache_dir=TRAIN_PART_CACHE_DIR,
        keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
        shuffle_parts=True,
        shuffle_examples=True,
        seed=RANK,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_streaming_protein_lm_dataloader(
            tokenizer_artifact.tokenizer,
            prefix_uri=MINIO_TRAIN_PARTS_PREFIX_URI or None,
            part_uris=MINIO_TRAIN_PART_URIS or None,
            config=MINIO_DATA_CONFIG,
            cache_dir=TRAIN_PART_CACHE_DIR,
            keep_downloaded_parts=KEEP_DOWNLOADED_TRAIN_PARTS,
            shuffle_parts=False,
            shuffle_examples=False,
            seed=0,
            distributed=False,
            **loader_kwargs,
        )
        if IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "minio streaming"
elif use_streaming_loader:
    train_loader = create_streaming_protein_lm_dataloader(
        tokenizer_artifact.tokenizer,
        part_paths=LOCAL_TRAIN_TEXT_PATHS,
        shuffle_parts=True,
        shuffle_examples=True,
        seed=RANK,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_streaming_protein_lm_dataloader(
            tokenizer_artifact.tokenizer,
            part_paths=LOCAL_TRAIN_TEXT_PATHS,
            shuffle_parts=False,
            shuffle_examples=False,
            seed=0,
            distributed=False,
            **loader_kwargs,
        )
        if IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "local part streaming"
else:
    train_loader = create_protein_lm_dataloader(
        train_text,
        tokenizer_artifact.tokenizer,
        shuffle=True,
        sampler_seed=0,
        **loader_kwargs,
        **distributed_loader_kwargs,
    )
    train_eval_loader = (
        create_protein_lm_dataloader(
            train_text,
            tokenizer_artifact.tokenizer,
            shuffle=False,
            distributed=False,
            **loader_kwargs,
        )
        if train_text and IS_MAIN_PROCESS
        else None
    )
    train_loader_description = "in-memory split"

val_loader = (
    create_protein_lm_dataloader(
        val_text,
        tokenizer_artifact.tokenizer,
        shuffle=False,
        distributed=False,
        **loader_kwargs,
    )
    if val_text and IS_MAIN_PROCESS
    else None
)

def dataloader_size(loader):
    if loader is None:
        return "disabled"
    try:
        return len(loader)
    except TypeError:
        return "streaming"

if IS_MAIN_PROCESS:
    corpus_chars = f"{len(corpus_text):,}" if corpus_text else "streaming/not loaded"
    print(f"Corpus chars: {corpus_chars}")
    print(f"Local corpus files: {len(LOCAL_TRAIN_TEXT_PATHS)}")
    print(f"Train loader: {train_loader_description}")
    print(f"Tokenizer vocab: {tokenizer_artifact.vocab_size} rebuilt={tokenizer_artifact.rebuilt}")
    print(f"Batches: train={dataloader_size(train_loader)} train_eval={dataloader_size(train_eval_loader)} val={dataloader_size(val_loader)}")
else:
    print(f"Rank {RANK}/{WORLD_SIZE} prepared distributed train loader: {train_loader_description}")

In [ ]:
requested_device = REQUESTED_DEVICE
if requested_device == "auto":
    requested_device = "cuda" if torch.cuda.is_available() else "cpu"
runtime_dtype = torch.bfloat16 if requested_device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

progen_config = build_progen_config(
    PROGEN_MODEL_SIZE,
    vocab_size=tokenizer_artifact.vocab_size,
    context_length=CONTEXT_LENGTH,
    dtype=runtime_dtype,
    overrides=PROGEN_CONFIG_OVERRIDES,
)

model_config = build_mdc_config_from_progen_config(
    progen_config,
    dtype=runtime_dtype,
    attention_pattern="as_config",
)
runtime = prepare_mdc_training_runtime(
    MDCDecoderModel(model_config),
    device=requested_device,
    multi_gpu=MULTI_GPU_MODE,
    find_unused_parameters=DDP_FIND_UNUSED_PARAMETERS,
    data_parallel_device_ids=DATA_PARALLEL_DEVICE_IDS,
)
model = runtime.model
device = runtime.device

optimizer = create_protein_training_optimizer(
    model,
    OPTIMIZER_CONFIG,
    device=device,
)
optimizer_names = describe_protein_training_optimizers(optimizer)

runtime_summary = {
    "device": str(device),
    "dtype": str(runtime_dtype),
    "distributed": runtime.distributed,
    "data_parallel": runtime.data_parallel,
    "rank": runtime.rank,
    "local_rank": runtime.local_rank,
    "world_size": runtime.world_size,
    "optimizer_types": optimizer_names,
    "trainable_parameters": count_trainable_parameters(model),
}

if runtime.is_main_process:
    print(f"Device: {device} dtype={runtime_dtype}")
    print(f"Distributed: {runtime.distributed} data_parallel={runtime.data_parallel} world_size={runtime.world_size}")
    print(f"Optimizers: {optimizer_names}")
    print(f"Trainable parameters: {count_trainable_parameters(model):,}")

runtime_summary

In [ ]:
checkpoint_last_path = CHECKPOINT_DIR / "checkpoint_last.pt"
checkpoint_best_path = CHECKPOINT_DIR / "checkpoint_best.pt"
metrics_path = CHECKPOINT_DIR / "training_metrics.jsonl"

global_step = 0
tokens_seen = 0
train_losses = []
val_losses = []
best_val_loss = math.inf
step_eval_enabled = EVAL_FREQ > 0 and not runtime.distributed and train_eval_loader is not None
optimizer_list = list(optimizer) if isinstance(optimizer, (list, tuple)) else [optimizer]

def move_batch(batch, device):
    return type(batch)(
        input_ids=batch.input_ids.to(device),
        attention_mask=batch.attention_mask.to(device),
        labels=batch.labels.to(device),
    )

def append_metrics(payload):
    with metrics_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

def distributed_barrier():
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        torch.distributed.barrier()

def count_step_tokens(batch):
    token_count = torch.tensor(
        int(batch.attention_mask.sum().item()),
        device=device,
        dtype=torch.long,
    )
    if runtime.distributed:
        torch.distributed.all_reduce(token_count, op=torch.distributed.ReduceOp.SUM)
    return int(token_count.item())

def save_checkpoint(path, epoch, val_loss):
    return save_protein_pretrain_checkpoint(
        path,
        model=model,
        optimizer=optimizer,
        model_config=model_config,
        tokenizer=tokenizer_artifact.tokenizer,
        tokenizer_map_path=tokenizer_artifact.tokenizer_map_path,
        epoch=epoch,
        global_step=global_step,
        tokens_seen=tokens_seen,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=None if math.isinf(best_val_loss) else best_val_loss,
        training_args={
            "batch_size": BATCH_SIZE,
            "context_length": CONTEXT_LENGTH,
            "stride": STRIDE,
            "learning_rate": LEARNING_RATE,
            "muon_learning_rate": MUON_LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer_type": OPTIMIZER_TYPE,
            "optimizer_types": optimizer_names,
            "progen_model_size": PROGEN_MODEL_SIZE,
            "progen_config_overrides": PROGEN_CONFIG_OVERRIDES,
            "multi_gpu_mode": MULTI_GPU_MODE,
            "num_workers": NUM_WORKERS,
            "pin_memory": PIN_MEMORY,
            "train_config_path": str(TRAINING_CONFIG["config_path"]),
        },
        extra={
            "corpus_file": str(TRAIN_TEXT_PATH.resolve()),
            "corpus_files": [str(path.resolve()) for path in LOCAL_TRAIN_TEXT_PATHS],
            "minio_train_parts_prefix_uri": MINIO_TRAIN_PARTS_PREFIX_URI,
            "minio_train_part_uris": list(MINIO_TRAIN_PART_URIS),
            "progen_config": dict(progen_config),
            "last_eval_val_loss": val_loss,
        },
    )

for epoch in range(1, NUM_EPOCHS + 1):
    set_mdc_data_loader_epoch(train_loader, epoch - 1)
    model.train()
    for batch in train_loader:
        batch = move_batch(batch, device)
        for opt in optimizer_list:
            opt.zero_grad(set_to_none=True)
        logits = model(batch.input_ids, batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        for opt in optimizer_list:
            opt.step()

        global_step += 1
        tokens_seen += count_step_tokens(batch)

        if step_eval_enabled and global_step % EVAL_FREQ == 0:
            train_eval_loss = evaluate_mdc_causal_lm_batch_loss(
                model,
                train_eval_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            val_loss = (
                evaluate_mdc_causal_lm_batch_loss(
                    model,
                    val_loader,
                    device=device,
                    max_batches=EVAL_BATCHES,
                )
                if val_loader is not None
                else float("nan")
            )
            train_losses.append(train_eval_loss)
            val_losses.append(val_loss)
            append_metrics({
                "epoch": epoch,
                "global_step": global_step,
                "tokens_seen": tokens_seen,
                "train_loss": train_eval_loss,
                "val_loss": val_loss,
            })
            if val_loader is not None and val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(checkpoint_best_path, epoch, val_loss)
            print(f"epoch={epoch} step={global_step} train={train_eval_loss:.4f} val={val_loss:.4f}")

    distributed_barrier()

    if runtime.is_main_process:
        train_eval_loss = (
            evaluate_mdc_causal_lm_batch_loss(
                model,
                train_eval_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            if train_eval_loader is not None
            else float("nan")
        )
        val_loss = (
            evaluate_mdc_causal_lm_batch_loss(
                model,
                val_loader,
                device=device,
                max_batches=EVAL_BATCHES,
            )
            if val_loader is not None
            else float("nan")
        )
        train_losses.append(train_eval_loss)
        val_losses.append(val_loss)
        append_metrics({
            "epoch": epoch,
            "global_step": global_step,
            "tokens_seen": tokens_seen,
            "train_loss": train_eval_loss,
            "val_loss": val_loss,
        })
        if val_loader is not None and val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(checkpoint_best_path, epoch, val_loss)
        save_checkpoint(checkpoint_last_path, epoch, val_loss)
        print(f"epoch={epoch} completed train={train_eval_loss:.4f} val={val_loss:.4f}")

    distributed_barrier()

if runtime.is_main_process:
    print(f"Saved last checkpoint: {checkpoint_last_path}")
    print(f"Saved best checkpoint: {checkpoint_best_path if checkpoint_best_path.exists() else 'not yet'}")
else:
    print(f"Rank {runtime.rank}/{runtime.world_size} finished epoch sync at global_step={global_step}")

In [ ]:
sample = (
    generate_protein_text(
        model,
        tokenizer_artifact.tokenizer,
        prompt="<|protein|>",
        device=device,
        max_new_tokens=80,
        context_length=CONTEXT_LENGTH,
    )
    if runtime.is_main_process
    else f"Sample generation is emitted on rank 0 only. This rank is {runtime.rank}."
)
sample